# FGAT - Notebook 1: Visual Feature Extraction
**Mục tiêu:** Tải ảnh từ URL → extract visual features bằng ResNet-152 → lưu ra `item_visual_embs.npy`

**Input:** `item_data.txt`, thư mục ảnh (tải từ URL)

**Output:** `/kaggle/working/item_visual_embs.npy` (shape: [N_items, 2048])

> **Kaggle GPU:** Bật Accelerator → GPU T4 x2 trong Settings

In [1]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'Pillow', 'tqdm'], check=True)
print('Done installing')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.0 MB/s eta 0:00:00
Done installing


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [2]:
# ============================================================
# CELL 2: Config - chỉnh sửa paths ở đây
# ============================================================
import os

# --- Kaggle: dataset POG phải được add vào notebook ---
# Vào Settings > Add Data > tìm POG dataset
DATA_DIR   = '//kaggle/input/datasets/kiettruonglifeez/recsys-fgat/'   # <-- đổi nếu tên dataset khác
ITEM_FILE  = os.path.join(DATA_DIR, 'item_data.txt')
IMAGE_DIR  = '/kaggle/working/images'       # ảnh sẽ được tải về đây
OUT_DIR    = '/kaggle/working'

os.makedirs(IMAGE_DIR, exist_ok=True)

VISUAL_DIM   = 2048   # ResNet-152 output dim
BATCH_SIZE   = 64     # số ảnh xử lý cùng lúc
MAX_WORKERS  = 8      # số thread tải ảnh song song
RETRY        = 3      # số lần retry nếu tải ảnh thất bại
TIMEOUT      = 10     # timeout (giây) mỗi request

print(f'Data dir : {DATA_DIR}')
print(f'Output   : {OUT_DIR}')

Data dir : //kaggle/input/datasets/kiettruonglifeez/recsys-fgat/
Output   : /kaggle/working


In [3]:
# ============================================================
# CELL 3: Load item_data
# ============================================================
import pandas as pd

item_data = pd.read_csv(ITEM_FILE, header=None,
                        names=['item_id', 'category', 'image_url', 'title'])
item_data['item_id'] = item_data['item_id'].astype(str)

print(f'Total items : {len(item_data)}')
print(item_data.head())

Total items : 19175
  item_id  category                                          image_url  \
0       0         3  http://gw.alicdn.com/imgextra/i3/18/O1CN01b5qP...   
1       1        10  http://gw.alicdn.com/imgextra/i4/67/O1CN010921...   
2       2        43  http://gw.alicdn.com/imgextra/i3/77/O1CN01I1Kw...   
3       3        16  http://gw.alicdn.com/imgextra/i4/190/O1CN01tu5...   
4       4        46  http://gw.alicdn.com/imgextra/i4/13/O1CN01kHo0...   

                                    title  
0  yeah jewelry官方店 原创设计首饰小人国耳钉925银镀金女礼物饰品  
1         20199秋冬季新款女装黑色修身长裙气质打底裙中长款毛呢连衣裙  
2         芍药居少女原创 万能打底裤女薄款经典高腰少女黑直筒紧身裤19春  
3          阿俪克斯高腰阔腿裤女春季2019新款刺绣街头条纹开叉运动裤子  
4        Sogo★个性卡通刺绣水洗做旧牛仔弯檐棒球帽男女情侣街头鸭舌帽潮  


In [4]:
# ============================================================
# CELL 4: Download images (parallel, skip if already exists)
# ============================================================
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/110.0.0.0 Safari/537.36'
}

def download_one(row):
    item_id, url = row['item_id'], row['image_url']
    save_path = os.path.join(IMAGE_DIR, f'{item_id}.png')
    if os.path.exists(save_path):
        return item_id, True   # already downloaded
    for attempt in range(RETRY):
        try:
            r = requests.get(url, headers=HEADERS, stream=True, timeout=TIMEOUT)
            if r.status_code == 200:
                with open(save_path, 'wb') as f:
                    for chunk in r.iter_content(1024):
                        f.write(chunk)
                return item_id, True
            time.sleep(2)
        except Exception:
            time.sleep(1)
    return item_id, False  # failed

rows = item_data.to_dict('records')
failed_ids = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_one, r): r['item_id'] for r in rows}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Downloading images'):
        item_id, ok = fut.result()
        if not ok:
            failed_ids.append(item_id)

print(f'Downloaded OK : {len(rows) - len(failed_ids)}/{len(rows)}')
print(f'Failed        : {len(failed_ids)}')

Downloaded OK : 19167/19175
Failed        : 8


In [5]:
# ============================================================
# CELL 5: Load ResNet-152 (feature extractor, no last FC)
# ============================================================
import torch
import torchvision.models as models
import torchvision.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

resnet152 = models.resnet152(pretrained=True)
resnet152 = torch.nn.Sequential(*list(resnet152.children())[:-1])  # remove FC
resnet152 = resnet152.to(device).eval()

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])
print('ResNet-152 loaded')

Device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth


100%|██████████| 230M/230M [00:01<00:00, 206MB/s]


ResNet-152 loaded


In [6]:
# ============================================================
# CELL 6: Extract visual features (batch mode)
# ============================================================
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class ImageDataset(Dataset):
    def __init__(self, item_ids, image_dir, transform):
        self.item_ids  = item_ids
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.item_ids)

    def __getitem__(self, idx):
        item_id  = self.item_ids[idx]
        img_path = os.path.join(self.image_dir, f'{item_id}.png')
        try:
            img = Image.open(img_path).convert('RGB')
            tensor = self.transform(img)
        except Exception:
            # fallback: zero tensor if image missing/corrupt
            tensor = torch.zeros(3, 224, 224)
        return item_id, tensor

all_item_ids = item_data['item_id'].tolist()
dataset      = ImageDataset(all_item_ids, IMAGE_DIR, transform)
loader       = DataLoader(dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, pin_memory=True)

visual_features = {}   # item_id -> np.array [2048]

with torch.no_grad():
    for batch_ids, batch_imgs in tqdm(loader, desc='Extracting visual features'):
        batch_imgs = batch_imgs.to(device)
        feats = resnet152(batch_imgs).squeeze(-1).squeeze(-1)  # [B, 2048]
        feats = feats.cpu().numpy().astype(np.float32)
        for item_id, feat in zip(batch_ids, feats):
            visual_features[item_id] = feat

print(f'Extracted features for {len(visual_features)} items')

Extracting visual features:   0%|          | 0/300 [00:00<?, ?it/s]

Extracted features for 19175 items


In [7]:
# ============================================================
# CELL 7: Save - ordered by item_data index
# (shape: [N_items, 2048], row i = item at index i)
# ============================================================

# Build ordered array
emb_array = np.zeros((len(all_item_ids), VISUAL_DIM), dtype=np.float32)
for i, item_id in enumerate(all_item_ids):
    if item_id in visual_features:
        emb_array[i] = visual_features[item_id]

# Save
out_path = os.path.join(OUT_DIR, 'item_visual_embs.npy')
np.save(out_path, emb_array)
print(f'Saved  : {out_path}')
print(f'Shape  : {emb_array.shape}')

# Also save item_id order (important for notebook 3)
id_order_path = os.path.join(OUT_DIR, 'item_id_order.npy')
np.save(id_order_path, np.array(all_item_ids))
print(f'Saved item_id order: {id_order_path}')

Saved  : /kaggle/working/item_visual_embs.npy
Shape  : (19175, 2048)
Saved item_id order: /kaggle/working/item_id_order.npy
